# 🌍 Industrial-Grade Plug-and-Play AI Analyst (Qwen2.5-3B)

This is the **Deep Training** version of the AI Analyst. 
**Upgrades:**
- ⌛ **1 Hour+ Training**: Increased epochs and steps for production-grade accuracy.
- 🛠️ **Plug-and-Play RAG**: Support for switching between different fintech databases.
- 📊 **1200+ Examples**: Massive dataset coverage for complex joints and accounting logic.
- 🛡️ **Self-Pruning**: Learns to handle irrelevant tables in the context.

In [ ]:
# 1. Install Requirements
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Load Qwen2.5-3B
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 
dtype = None 
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 4. Data Loading
Upload your `dataset_pro.jsonl`.

In [ ]:
from datasets import load_dataset
import os

if os.path.exists("dataset_pro.jsonl"):
    dataset = load_dataset("json", data_files="dataset_pro.jsonl", split="train")
    
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an Industrial-Grade AI Data Analyst. Generate accurate MySQL for the given question.
Question: {}

### Input (Schema Context):
{}

### Response (MySQL Query):
{}"""

    EOS_TOKEN = tokenizer.eos_token
    def formatting_prompts_func(examples):
        instructions = examples["instruction"]
        inputs       = examples["input"]
        outputs      = examples["output"]
        texts = []
        for instruction, input, output in zip(instructions, inputs, outputs):
            text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
            texts.append(text)
        return { "text" : texts, }

    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"Dataset loaded with {len(dataset)} examples.")
else:
    print("ERROR: Please upload 'dataset_pro.jsonl' first!")

In [ ]:
# 5. Deep Training (Target: 1 Hour)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 10, # 10 Epochs for expert-level knowledge
        learning_rate = 1e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.05,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no",
    ),
)

trainer_stats = trainer.train()

### 6. Test Inference
**NOTE:** We have disabled `FastLanguageModel.for_inference` here to prevent broadcast shape errors in the current Unsloth build. This is slightly slower but 100% reliable.

In [ ]:
import torch
model.config.use_cache = True 

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Total GL entries vs loan disbursements for Branch A this month.",
        "Tables: acc_gl_journal_entry, m_loan, m_office. Columns: j.amount, l.principal_disbursed_derived, o.name",
        "", 
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

# Standard generation without the inference patch
with torch.no_grad():
    _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256, use_cache = True)

### 7. Global Export (GGUF)

In [ ]:
model.save_pretrained_gguf("industrial_sql_qwen_gguf", tokenizer, quantization_method = "q4_k_m")

In [ ]:
from google.colab import files
!zip -r industrial_model.zip industrial_sql_qwen_gguf
files.download("industrial_model.zip")